# Context and State

Context: immediate, turn-level working set of information the agent sees during a single reasoning step. It's the agent's real-time "view of the world" and contains specific and relevant information an agent has about a question or task that will improve the accuracy and reliability of its answers.

State: structured, persistent data and AI agent maintains across execution steps and sessions. It's the agent's durable memory, enabling continuity by remembering what happened in previous sessions.

In [10]:
from langchain.agents import create_agent, AgentState
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver

from dotenv import load_dotenv

from dataclasses import dataclass

from pprint import pprint

## Context

Context is only passed during tool calls at runtime, not when the agent is configured. Therefore, adding context to the agent requires four steps:
- Create a dataclass with the appropriate context variables
- Configure tools to utilize or retrieve context
- Pass the context schema to the agent when it is created
- Pass an instance of the context class to the agent when it is invoked

#### Dataclass

In [14]:
@dataclass
class ColorContext:
    favorite_color: str = "blue"
    least_favorite_color: str = "orange"

#### Tools

In [ ]:
@tool
def get_favorite_color(runtime: ToolRuntime) -> str:
    """Get the favorite color of the user"""
    return runtime.context.favorite_color

@tool
def get_least_favorite_color(runtime: ToolRuntime) -> str:
    """Get the least favorite color of the user"""
    return runtime.context.least_favorite_color

#### Creating the Agent

In [24]:
agent = create_agent(
    model='claude-haiku-4-5',
    context_schema=ColorContext,
    tools=[get_favorite_color, get_least_favorite_color]
)

#### Implementing

In [25]:
msg = HumanMessage(content="What is my least favorite color?")

response = agent.invoke({'messages': [msg]},
                        context=ColorContext(),
                        tools=[get_favorite_color, get_least_favorite_color])

/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favorite_col...favorite_color='orange'), input_type=ColorContext])
  function=lambda v, h: h(v), schema=original_schema
/Users/samueljoseph/Documents/Programming/agent-lab/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(favorite_col...favorite_color='orange'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


In [26]:
response

{'messages': [HumanMessage(content='What is my least favorite color?', additional_kwargs={}, response_metadata={}, id='d2c06613-667e-4958-9847-e1e38389d840'),
  AIMessage(content=[{'id': 'toolu_01B7JT4pYg6y9GoLA3fQPWf1', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'get_least_favorite_color', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Ce6jKG5fwzFGXVKffbXBU', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 600, 'output_tokens': 41, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a00b5e-6196-7c80-b6d6-eaf8a3122245-0', tool_calls=[{'name': 'g

## State

Implementation of agent state is done in three steps:
- Configuring a custom state class that inherits from langgraph's AgentState
- Creating a tool that updates the agent's state
- Creating a tool that can access the agent's state at runtime
- Initializing the agent with the updating tool, memory and the state schema

In [ ]:
# A class for setting the custom agent state
class CustomState(AgentState):
    favorite_color: str

In [14]:
# A tool that updates the custom agent state
@tool
def update_favorite_color(favorite_color: str, runtime: ToolRuntime) -> Command:
    """Update the favorite color of the user in the state once they've revealed it"""
    return Command[tuple[()]](update={
        "favorite_color": favorite_color,
        "messages": [ToolMessage("Successfully updated favorite color",
                                 tool_call_id=runtime.tool_call_id)]
    })

# A tool that accesses the agent's state
@tool
def read_favorite_color(runtime: ToolRuntime) -> str:
    """Read the favorite color of the user from the state"""
    try:
        return runtime.state["favorite_color"]
    except:
        return "No favorite color found in state"

In [ ]:
stateful_agent = create_agent(
    model='claude-haiku-4-5',
    tools=[update_favorite_color, read_favorite_color],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [24]:
response = stateful_agent.invoke(
    {"messages": [HumanMessage(content="My favorite color is blue")]},
    {"configurable": {"thread_id": "1"}}
)

In [25]:
pprint(response)

{'favorite_color': 'blue',
 'messages': [HumanMessage(content='My favorite color is blue', additional_kwargs={}, response_metadata={}, id='fa29e806-5499-46f6-9cad-8c09ce50578a'),
              AIMessage(content=[{'id': 'toolu_01AnUHK9e8HizUYh1TbC3itS', 'caller': {'type': 'direct'}, 'input': {'favorite_color': 'blue'}, 'name': 'update_favorite_color', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Ce8FJBfEH3ytjLJ9J7hBF', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 623, 'output_tokens': 58, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a00f7e-ac

In [26]:
response2 = stateful_agent.invoke(
    {"messages": [HumanMessage(content="What is my favorite color?")]},
    {"configurable": {"thread_id": "1"}}
)

In [27]:
pprint(response2)

{'favorite_color': 'blue',
 'messages': [HumanMessage(content='My favorite color is blue', additional_kwargs={}, response_metadata={}, id='fa29e806-5499-46f6-9cad-8c09ce50578a'),
              AIMessage(content=[{'id': 'toolu_01AnUHK9e8HizUYh1TbC3itS', 'caller': {'type': 'direct'}, 'input': {'favorite_color': 'blue'}, 'name': 'update_favorite_color', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Ce8FJBfEH3ytjLJ9J7hBF', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 623, 'output_tokens': 58, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a00f7e-ac